In [1]:
# ==============================================================================
# CELL 1: BIGQUERY AUTH & REALITY INGEST (SOP 0407 - CODESPACES MODE)
# ==============================================================================
import os
import pandas as pd
from google.cloud import bigquery
from google.oauth2 import service_account
from IPython.display import display, Markdown
import warnings
warnings.filterwarnings('ignore')

# --- 1. CONFIGURACIÓN DE IDENTIDAD ---
PROJECT_ID = 'drivers-dilemma'
# Ruta confirmada por el Arquitecto
SERVICE_ACCOUNT_JSON = '/workspaces/pienza/secrets/service-account.json' 

display(Markdown(f"### 🚀 **Iniciando Downscaling (BigQuery Mode: {PROJECT_ID})**"))

if not os.path.exists(SERVICE_ACCOUNT_JSON):
    raise FileNotFoundError(f"🔴 ERROR: No se encontró la llave en: {SERVICE_ACCOUNT_JSON}. Verifica que el archivo esté en la carpeta /secrets.")

# --- 2. AUTENTICACIÓN Y CLIENTE ---
# Usamos las credenciales explícitas para evitar depender de gcloud auth login
credentials = service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_JSON)
client = bigquery.Client(credentials=credentials, project=PROJECT_ID)
print("✅ Conexión establecida con Google Cloud via Service Account.")

# --- 3. INGESTA SINTÉTICA (GAN) ---
print("⏳ Descargando Manifold Sintético Completo (1M+ viajes)...")
query_gan = f"SELECT * FROM `{PROJECT_ID}.pienza_big.synthetic_manifold_v8_enriched`"
df_synthetic = client.query(query_gan).to_dataframe()
print(f"   ✅ GAN descargado: {len(df_synthetic):,} registros sintéticos.")

# --- 4. INGESTA DE REALIDAD (v_ML_Supervised) ---
print("⏳ Descargando Realidad Absoluta para Downscaling...")
query_real = f"""
    SELECT
        pickup_lat, pickup_lon,
        dropoff_polygon_id, dropoff_polygon_name,
        dropoff_hdbscan_id, dropoff_hdbscan_name
    FROM `{PROJECT_ID}.pienza_mini.v_ML_Supervised`
"""
df_real = client.query(query_real).to_dataframe()

# Sanitización de tipos para paridad de datos
df_real['dropoff_polygon_id'] = pd.to_numeric(df_real['dropoff_polygon_id'], errors='coerce')
df_real['dropoff_hdbscan_id'] = pd.to_numeric(df_real['dropoff_hdbscan_id'], errors='coerce')

print(f"   ✅ Realidad descargada: {len(df_real):,} registros base.")
print("-" * 65)

### 🚀 **Iniciando Downscaling (BigQuery Mode: drivers-dilemma)**

✅ Conexión establecida con Google Cloud via Service Account.
⏳ Descargando Manifold Sintético Completo (1M+ viajes)...
   ✅ GAN descargado: 1,010,001 registros sintéticos.
⏳ Descargando Realidad Absoluta para Downscaling...
   ✅ Realidad descargada: 4,765 registros base.
-----------------------------------------------------------------


In [3]:
# ==============================================================================
# CELL 2: RECONSTRUCCIÓN DE IDENTIDADES MICRO (PATCH: REFORMA SOCIAL - FIXED)
# ==============================================================================
import geopandas as gpd
import numpy as np
import pandas as pd

print("🧮 RECONSTRUYENDO IDs Y NOMBRES (DESAMBIGUACIÓN TOPOLÓGICA)...")

# --- 1. DICCIONARIO MAESTRO (Sin cambios) ---
id_map = {
    -1:-1, 41:0, 42:0, 46:0, 43:1, 65:2, 62:2, 44:2, 36:2, 49:3, 52:3, 35:3, 50:4, 58:4,
    25:5, 31:5, 63:6, 39:6, 51:7, 33:7, 37:8, 53:8, 48:8, 60:9, 57:10, 12:10, 32:10,
    24:11, 40:12, 45:13, 59:13, 61:14, 38:14, 34:15, 30:16, 66:16, 17:17, 14:17, 22:17,
    16:18, 13:18, 11:19, 15:20, 21:21, 20:21, 19:21, 18:22, 47:23, 55:23, 56:23, 54:24,
    64:24, 71:25, 9:26, 70:27, 69:28, 8:29, 6:30, 7:30, 23:30, 3:31, 2:32, 4:33, 29:33,
    68:34, 5:35, 27:36, 28:36, 1:37, 10:38, 0:39, 26:40, 67:41
}

# --- 2. DROPOFFS ---
df_real.loc[df_real['dropoff_polygon_id'] == 19, 'dropoff_polygon_name'] = 'reforma_social'
df_real.loc[df_real['dropoff_polygon_id'] == 66, 'dropoff_polygon_name'] = 'tecamachalco'
df_real['id_agrupado_drop'] = df_real['dropoff_polygon_id'].fillna(-1).astype(int).map(id_map).fillna(-1).astype(int)

cond_drop = [(df_real['id_agrupado_drop'] >= 0), (df_real['dropoff_hdbscan_id'] > -1)]
choice_id = ["P_" + df_real['id_agrupado_drop'].astype(str), "C_" + df_real['dropoff_hdbscan_id'].fillna(-1).astype(int).astype(str)]
df_real['dropoff_zone_id'] = np.select(cond_drop, choice_id, default="Unassigned")
df_real['dropoff_micro_name'] = np.select(cond_drop, [df_real['dropoff_polygon_name'], df_real['dropoff_hdbscan_name']], default="unassigned_area")

# --- 3. PICKUPS (FIXED COORDINATE ALIGNMENT) ---
print("⏳ Ejecutando SJoin Espacial...")

# A. Limpieza de NaNs ANTES de generar la geometría
df_pickups_clean = df_real.dropna(subset=['pickup_lat', 'pickup_lon']).copy()

# B. Generación de puntos sincronizada con la tabla limpia
gdf_pickups = gpd.GeoDataFrame(
    df_pickups_clean, 
    geometry=gpd.points_from_xy(df_pickups_clean.pickup_lon, df_pickups_clean.pickup_lat), 
    crs="EPSG:4326"
)

# C. Carga y Parche de GeoJSON
gdf_polygons = gpd.read_file("/workspaces/pienza/assets/poly.geojson").to_crs("EPSG:4326")
tecas = gdf_polygons[gdf_polygons['name'].str.lower().str.strip() == 'tecamachalco']
if len(tecas) > 1:
    idx_este = tecas.geometry.centroid.x.idxmax()
    idx_oeste = tecas.geometry.centroid.x.idxmin()
    gdf_polygons.at[idx_este, 'name'] = 'reforma_social'
    gdf_polygons.at[idx_oeste, 'name'] = 'tecamachalco'

# D. Spatial Join
joined = gpd.sjoin(gdf_pickups, gdf_polygons[['geometry', 'name']], how="left", predicate='within')

# --- 🟢 EL FIX PARA LOS "VIAJES EXTRA" 🟢 ---
# Eliminamos duplicados basados en el índice original para no inflar la realidad
joined = joined[~joined.index.duplicated(keep='first')]

joined['pickup_agrupado'] = joined['index_right'].map(id_map).fillna(-1).astype(int)
joined['pickup_zone_id'] = np.where(joined['pickup_agrupado'] >= 0, "P_" + joined['pickup_agrupado'].astype(str), "Unassigned")
joined['pickup_micro_name'] = joined['name'].fillna('unassigned_area').astype(str).str.strip().str.lower()

print(f"✅ SJoin completado y podado: {len(joined)} viajes (Paridad total con df_real).")

🧮 RECONSTRUYENDO IDs Y NOMBRES (DESAMBIGUACIÓN TOPOLÓGICA)...
⏳ Ejecutando SJoin Espacial...
✅ SJoin completado y podado: 4762 viajes (Paridad total con df_real).


In [4]:
# ==============================================================================
# Cell 3: Motor de Downscaling (Outer Join + Redondeo Blindado) - FIXED
# ==============================================================================
import pandas as pd
import numpy as np

print("⚖️ INICIANDO DOWNSCALING MATEMÁTICO...")
print("-" * 65)

# --- 0. BRIDGE: Creación de tablas de agregación (Paridad Nativa) ---
# Estas son las que faltaban en memoria
df_mini_dropoffs = df_real.groupby(['dropoff_zone_id', 'dropoff_micro_name']).size().reset_index(name='hist_dropoffs')
df_mini_pickups = joined.groupby(['pickup_zone_id', 'pickup_micro_name']).size().reset_index(name='hist_pickups')

# --- 1. PICKUPS ---
df_mini_pickups['macro_total'] = df_mini_pickups.groupby('pickup_zone_id')['hist_pickups'].transform('sum')
df_mini_pickups['weight'] = df_mini_pickups['hist_pickups'] / df_mini_pickups['macro_total']

gan_p = df_synthetic.groupby('pickup_zone_id').size().reset_index(name='gan_pickups')
TARGET_P = gan_p['gan_pickups'].sum() 

downscale_p = pd.merge(df_mini_pickups, gan_p, on='pickup_zone_id', how='outer')
downscale_p['weight'] = downscale_p['weight'].fillna(1.0)
downscale_p['pickup_micro_name'] = downscale_p['pickup_micro_name'].fillna(downscale_p['pickup_zone_id'])
downscale_p['gan_pickups'] = downscale_p['gan_pickups'].fillna(0)

# Multiplicación y Redondeo
downscale_p['micro_gan_pickups'] = (downscale_p['gan_pickups'] * downscale_p['weight'])
downscale_p['micro_gan_pickups'] = downscale_p['micro_gan_pickups'].round(0).fillna(0).astype(int)

# Ajuste de diferencias por redondeo
diff_p = int(TARGET_P - downscale_p['micro_gan_pickups'].sum())
if diff_p != 0:
    max_idx = downscale_p['micro_gan_pickups'].idxmax()
    downscale_p.loc[max_idx, 'micro_gan_pickups'] += diff_p

# --- 2. DROPOFFS ---
df_mini_dropoffs['macro_total'] = df_mini_dropoffs.groupby('dropoff_zone_id')['hist_dropoffs'].transform('sum')
df_mini_dropoffs['weight'] = df_mini_dropoffs['hist_dropoffs'] / df_mini_dropoffs['macro_total']

gan_d = df_synthetic.groupby('dropoff_zone_id').size().reset_index(name='gan_dropoffs')
TARGET_D = gan_d['gan_dropoffs'].sum() 

downscale_d = pd.merge(df_mini_dropoffs, gan_d, on='dropoff_zone_id', how='outer')
downscale_d['weight'] = downscale_d['weight'].fillna(1.0)
downscale_d['dropoff_micro_name'] = downscale_d['dropoff_micro_name'].fillna(downscale_d['dropoff_zone_id'])
downscale_d['gan_dropoffs'] = downscale_d['gan_dropoffs'].fillna(0)

# Multiplicación y Redondeo
downscale_d['micro_gan_dropoffs'] = (downscale_d['gan_dropoffs'] * downscale_d['weight'])
downscale_d['micro_gan_dropoffs'] = downscale_d['micro_gan_dropoffs'].round(0).fillna(0).astype(int)

# Ajuste de diferencias por redondeo
diff_d = int(TARGET_D - downscale_d['micro_gan_dropoffs'].sum())
if diff_d != 0:
    max_idx = downscale_d['micro_gan_dropoffs'].idxmax()
    downscale_d.loc[max_idx, 'micro_gan_dropoffs'] += diff_d

print(f"🏆 VOLUMEN REPARTIDO Y REDONDEADO:")
print(f"    -> Pickups:  {downscale_p['micro_gan_pickups'].sum():,.0f} (Target: {TARGET_P:,.0f})")
print(f"    -> Dropoffs: {downscale_d['micro_gan_dropoffs'].sum():,.0f} (Target: {TARGET_D:,.0f})")

⚖️ INICIANDO DOWNSCALING MATEMÁTICO...
-----------------------------------------------------------------
🏆 VOLUMEN REPARTIDO Y REDONDEADO:
    -> Pickups:  1,010,001 (Target: 1,010,001)
    -> Dropoffs: 1,010,001 (Target: 1,010,001)


In [5]:
# ==============================================================================
# Cell 4: Auditoría de Fidelidad 1 vs 1
# ==============================================================================
def format_audit(df, name_col, hist_col, gan_col):
    audit = df[[name_col, hist_col, gan_col]].copy()

    # Agrupamos por Micro Name (Por si el outer join duplicó algo)
    audit = audit.groupby(name_col).sum().reset_index()

    t_hist = audit[hist_col].sum()
    t_gan = audit[gan_col].sum()

    audit['share_real'] = (audit[hist_col] / t_hist) * 100 if t_hist > 0 else 0
    audit['share_synth'] = (audit[gan_col] / t_gan) * 100 if t_gan > 0 else 0
    audit['delta_abs'] = audit['share_synth'] - audit['share_real']

    mae = audit['delta_abs'].abs().mean()

    audit['share_real'] = audit['share_real'].map('{:.4f} %'.format)
    audit['share_synth'] = audit['share_synth'].map('{:.4f} %'.format)
    audit['delta_abs'] = audit['delta_abs'].map('{:+.4f} %'.format)

    return audit.sort_values(by=name_col).reset_index(drop=True), mae

audit_p_df, mae_p = format_audit(downscale_p, 'pickup_micro_name', 'hist_pickups', 'micro_gan_pickups')
audit_d_df, mae_d = format_audit(downscale_d, 'dropoff_micro_name', 'hist_dropoffs', 'micro_gan_dropoffs')

print("📋 ESCÁNER DE FIDELIDAD 1 VS 1 (SHARE %)")
print("-" * 85)
print(f"📍 AUDITORÍA DE ORÍGENES (PICKUPS) | MAE: {mae_p:.6f} %")
with pd.option_context('display.max_rows', 100): display(audit_p_df)

print("-" * 85)
print(f"🏁 AUDITORÍA DE DESTINOS (DROPOFFS) | MAE: {mae_d:.6f} %")
with pd.option_context('display.max_rows', 100): display(audit_d_df)

📋 ESCÁNER DE FIDELIDAD 1 VS 1 (SHARE %)
-------------------------------------------------------------------------------------
📍 AUDITORÍA DE ORÍGENES (PICKUPS) | MAE: 0.125593 %


,pickup_micro_name,hist_pickups,micro_gan_pickups,share_real,share_synth,delta_abs
0,agwa_bezares,30,5655,0.6300 %,0.5599 %,-0.0701 %
1,ahuehuetes_norte,36,6761,0.7560 %,0.6694 %,-0.0866 %
2,ahuehuetes_sur,71,16684,1.4910 %,1.6519 %,+0.1609 %
3,anahuac_1,24,4958,0.5040 %,0.4909 %,-0.0131 %
4,anzures,303,68939,6.3629 %,6.8256 %,+0.4628 %
5,ave_club_de_golf_lomas,2,487,0.0420 %,0.0482 %,+0.0062 %
6,bahias,85,17561,1.7850 %,1.7387 %,-0.0463 %
7,blvrd_anahuac,19,3577,0.3990 %,0.3542 %,-0.0448 %
8,bondojito_asf,6,1162,0.1260 %,0.1150 %,-0.0109 %
9,bosque_2,10,1937,0.2100 %,0.1918 %,-0.0182 %


-------------------------------------------------------------------------------------
🏁 AUDITORÍA DE DESTINOS (DROPOFFS) | MAE: 0.118682 %


,dropoff_micro_name,hist_dropoffs,micro_gan_dropoffs,share_real,share_synth,delta_abs
0,agwa_bezares,19,3644,0.3987 %,0.3608 %,-0.0379 %
1,ahuehuetes_norte,19,3587,0.3987 %,0.3551 %,-0.0436 %
2,ahuehuetes_sur,48,11857,1.0073 %,1.1740 %,+0.1666 %
3,anahuac_1,24,5723,0.5037 %,0.5666 %,+0.0630 %
4,anzures,48,9282,1.0073 %,0.9190 %,-0.0883 %
5,ave_club_de_golf_lomas,3,531,0.0630 %,0.0526 %,-0.0104 %
6,bahias,15,3577,0.3148 %,0.3542 %,+0.0394 %
7,barranca_del_muerto,32,5922,0.6716 %,0.5863 %,-0.0852 %
8,blvrd_anahuac,12,3205,0.2518 %,0.3173 %,+0.0655 %
9,bondojito_asf,13,3354,0.2728 %,0.3321 %,+0.0593 %


In [10]:
# ==============================================================================
# Cell 5: Inyección de Nombres Micro y Persistencia (Versión Anti-Duplicados)
# ==============================================================================
import os
import pandas as pd

print("📦 FORJANDO EL MANIFOLD DEFINITIVO (DOWN-MAPPED)...")
print("-" * 65)

# --- 0. LIMPIEZA PREVENTIVA ---
# Si re-corriste la celda y ya había duplicados, los borramos de entrada
df_synthetic = df_synthetic.loc[:, ~df_synthetic.columns.duplicated()].copy()

# --- 1. CREACIÓN DE LOS POOLS DE IDENTIDAD ---
print("🎲 Generando pools de nombres estocásticos...")
pickup_pool = downscale_p.loc[downscale_p.index.repeat(downscale_p['micro_gan_pickups'])].copy()
pickup_pool = pickup_pool[['pickup_zone_id', 'pickup_micro_name']].sample(frac=1).reset_index(drop=True)

dropoff_pool = downscale_d.loc[downscale_d.index.repeat(downscale_d['micro_gan_dropoffs'])].copy()
dropoff_pool = dropoff_pool[['dropoff_zone_id', 'dropoff_micro_name']].sample(frac=1).reset_index(drop=True)

# --- 2. MAPEO POR REEMPLAZO SECUENCIAL ---
print("💉 Inyectando identidades micro al millón de viajes...")
p_col = 'pickup_zone_id' if 'pickup_zone_id' in df_synthetic.columns else 'pickup_id_GAN'
d_col = 'dropoff_zone_id' if 'dropoff_zone_id' in df_synthetic.columns else 'dropoff_id_GAN'

df_synthetic = df_synthetic.sort_values(p_col).reset_index(drop=True)
pickup_pool = pickup_pool.sort_values('pickup_zone_id').reset_index(drop=True)
df_synthetic['pickup_micro_name'] = pickup_pool['pickup_micro_name'].values

df_synthetic = df_synthetic.sort_values(d_col).reset_index(drop=True)
dropoff_pool = dropoff_pool.sort_values('dropoff_zone_id').reset_index(drop=True)
df_synthetic['dropoff_micro_name'] = dropoff_pool['dropoff_micro_name'].values

# --- 3. MEZCLA FINAL ---
df_synthetic = df_synthetic.sample(frac=1).reset_index(drop=True)

# --- 4. RENOMBRADO Y REORDENAMIENTO SEGURO ---
print("🏷️ Aplicando nueva nomenclatura (GAN vs Downscaled)...")
rename_dict = {
    'dropoff_zone_id': 'dropoff_id_GAN',
    'pickup_zone_id': 'pickup_id_GAN',
    'dropoff_name': 'dropoff_name_GAN',
    'pickup_name': 'pickup_name_GAN',
    'pickup_micro_name': 'pickup_name_down',
    'dropoff_micro_name': 'dropoff_name_down'
}

actual_rename = {k: v for k, v in rename_dict.items() if k in df_synthetic.columns}
df_synthetic = df_synthetic.rename(columns=actual_rename)

# 🔥 EL FILTRO ANTI-DUPLICADOS CRÍTICO 🔥
# Esto asegura que si una columna se llamó igual por error, solo quede la última versión
df_synthetic = df_synthetic.loc[:, ~df_synthetic.columns.duplicated()]

final_cols = [
    'upfront_fare', 'est_trip_time_sec', 'est_trip_dist_km',
    'time_to_pickup_sec', 'dist_to_pickup_km', 'hour_of_day',
    'day_of_week', 'product_category_fk', 'dropoff_id_GAN',
    'pickup_id_GAN', 'reason_primary_fk', 'product_name',
    'dropoff_name_GAN', 'pickup_name_GAN', 'pickup_name_down',
    'dropoff_name_down'
]

existing_cols = [c for c in final_cols if c in df_synthetic.columns]
remaining_cols = [c for c in df_synthetic.columns if c not in existing_cols]
df_synthetic = df_synthetic[existing_cols + remaining_cols]

# --- 5. GUARDADO EN DISCO (LOCAL CODESPACES - SOP 06XX) ---
DUMP_DIR = "/workspaces/pienza/data/dumped_files"
SOP_PREFIX = "06XX"
SAVE_PATH = f"{DUMP_DIR}/{SOP_PREFIX}_260503_synthetic_manifold_v8_downscaled.parquet"

os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
print(f"\n💾 Guardando Parquet definitivo en: {SAVE_PATH}")
df_synthetic.to_parquet(SAVE_PATH, index=False)

file_size = os.path.getsize(SAVE_PATH) / (1024 * 1024)
print(f"✅ ARCHIVO ASEGURADO: {file_size:.2f} MB")
print("-" * 65)
print("🏁 EL MOTOR DE DOWNSCALING HA TERMINADO. LA ARENA ESTÁ LISTA.")

📦 FORJANDO EL MANIFOLD DEFINITIVO (DOWN-MAPPED)...
-----------------------------------------------------------------
🎲 Generando pools de nombres estocásticos...
💉 Inyectando identidades micro al millón de viajes...
🏷️ Aplicando nueva nomenclatura (GAN vs Downscaled)...

💾 Guardando Parquet definitivo en: /workspaces/pienza/data/dumped_files/06XX_260503_synthetic_manifold_v8_downscaled.parquet
✅ ARCHIVO ASEGURADO: 36.60 MB
-----------------------------------------------------------------
🏁 EL MOTOR DE DOWNSCALING HA TERMINADO. LA ARENA ESTÁ LISTA.


In [11]:
# ==============================================================================
# Cell 1.6: Auditoría Post-Desambiguación (Estado en Memoria)
# Objetivo: Validar que 'reforma_social' y 'tecamachalco' coexistan en gdf_polygons
# ==============================================================================

print("🔍 AUDITANDO RESULTADO DE LA DESAMBIGUACIÓN (IN-MEMORY)")
print("-" * 65)

# 1. Verificación directa de existencia de nombres
target_names = ['reforma_social', 'tecamachalco']
found_names = gdf_polygons[gdf_polygons['name'].isin(target_names)]

print(f"📊 Reporte de Polígonos Críticos:")
if not found_names.empty:
    for name in target_names:
        count = len(gdf_polygons[gdf_polygons['name'] == name])
        print(f"   -> '{name}': {count} polígono(s) encontrado(s).")
else:
    print("   ⚠️ Error: No se encontraron los nombres esperados.")

print("-" * 65)

# 2. Análisis de Duplicados General
name_counts_final = gdf_polygons['name'].value_counts()
dupes_final = name_counts_final[name_counts_final > 1]

if dupes_final.empty:
    print("✅ LIMPIEZA TOTAL: No existen nombres duplicados en gdf_polygons.")
else:
    print(f"⚠️ Atención: Aún persisten duplicados en: {dupes_final.index.tolist()}")

# 3. Muestra de Coordenadas Finales
if len(found_names) >= 2:
    print("\n📍 Coordenadas Finales (Verificación de Eje X):")
    for idx, row in found_names.iterrows():
        print(f"   - {row['name'].ljust(15)} | Longitud (X): {row.geometry.centroid.x:.5f}")

print("-" * 65)

🔍 AUDITANDO RESULTADO DE LA DESAMBIGUACIÓN (IN-MEMORY)
-----------------------------------------------------------------
📊 Reporte de Polígonos Críticos:
   -> 'reforma_social': 1 polígono(s) encontrado(s).
   -> 'tecamachalco': 1 polígono(s) encontrado(s).
-----------------------------------------------------------------
✅ LIMPIEZA TOTAL: No existen nombres duplicados en gdf_polygons.

📍 Coordenadas Finales (Verificación de Eje X):
   - reforma_social  | Longitud (X): -99.22069
   - tecamachalco    | Longitud (X): -99.22957
-----------------------------------------------------------------


In [12]:
# ==============================================================================
# Cell 6: Auditoría Específica - El Duelo: Reforma Social vs. Tecamachalco
# Objetivo: Validar paridad y realismo en el reparto de los nodos desambiguados.
# ==============================================================================

print("🎯 AUDITORÍA DE PRECISIÓN: REFORMA SOCIAL VS. TECAMACHALCO")
print("-" * 85)

targets = ['reforma_social', 'tecamachalco']

# --- 1. AUDITORÍA PICKUPS (Basado en el SJOIN desambiguado) ---
p_audit = downscale_p[downscale_p['pickup_micro_name'].isin(targets)].copy()
p_audit['share_in_group'] = (p_audit['micro_gan_pickups'] / p_audit['micro_gan_pickups'].sum()) * 100

print("📍 ORIGENES (PICKUPS) - Basado en Geometría Real:")
if not p_audit.empty:
    display(p_audit[['pickup_micro_name', 'hist_pickups', 'micro_gan_pickups', 'share_in_group']]
            .sort_values('pickup_micro_name'))
else:
    print("⚠️ No se encontraron datos para estos nodos en Pickups.")

print("\n" + "-" * 85)

# --- 2. AUDITORÍA DROPOFFS (Basado en ID_Map + Nomenclatura Real) ---
d_audit = downscale_d[downscale_d['dropoff_micro_name'].isin(targets)].copy()
d_audit['share_in_group'] = (d_audit['micro_gan_dropoffs'] / d_audit['micro_gan_dropoffs'].sum()) * 100

print("🏁 DESTINOS (DROPOFFS) - Basado en Etiquetado Histórico:")
if not d_audit.empty:
    display(d_audit[['dropoff_micro_name', 'hist_dropoffs', 'micro_gan_dropoffs', 'share_in_group']]
            .sort_values('dropoff_micro_name'))
else:
    print("⚠️ No se encontraron datos para estos nodos en Dropoffs.")

print("-" * 85)
print("💡 NOTA: Si en Dropoffs 'reforma_social' aparece en 0, es porque el log histórico")
print("   de Uber (v_ML_Supervised) los etiqueta a ambos como 'tecamachalco'.")

🎯 AUDITORÍA DE PRECISIÓN: REFORMA SOCIAL VS. TECAMACHALCO
-------------------------------------------------------------------------------------
📍 ORIGENES (PICKUPS) - Basado en Geometría Real:


,pickup_micro_name,hist_pickups,micro_gan_pickups,share_in_group
26,reforma_social,13,2505,43.093067
14,tecamachalco,18,3308,56.906933



-------------------------------------------------------------------------------------
🏁 DESTINOS (DROPOFFS) - Basado en Etiquetado Histórico:


,dropoff_micro_name,hist_dropoffs,micro_gan_dropoffs,share_in_group
52,reforma_social,8,2264,62.784248
39,tecamachalco,7,1342,37.215752


-------------------------------------------------------------------------------------
💡 NOTA: Si en Dropoffs 'reforma_social' aparece en 0, es porque el log histórico
   de Uber (v_ML_Supervised) los etiqueta a ambos como 'tecamachalco'.


# La siguiente celda sube el archivo a GCS y crea una tabla externa en BigQuery


In [14]:
import os
from google.cloud import storage, bigquery
from google.oauth2 import service_account

# --- 1. CONFIGURACIÓN ---
PROJECT_ID = "drivers-dilemma"
SERVICE_ACCOUNT_JSON = '/workspaces/pienza/secrets/service-account.json'
GCS_BUCKET = "pienza-streamlit"

# EL ARCHIVO QUE ACABAMOS DE GENERAR
LOCAL_FILE_PATH = "/workspaces/pienza/data/dumped_files/06XX_260503_synthetic_manifold_v8_downscaled.parquet"
FILE_NAME = os.path.basename(LOCAL_FILE_PATH) # Esto extrae "06XX_260503_..."
GCS_URI = f"gs://{GCS_BUCKET}/{FILE_NAME}"

# --- 2. CREDENCIALES ---
credentials = service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_JSON)

# --- 3. PASO 1: SUBIR EL ARCHIVO A CLOUD STORAGE ---
print(f"📦 Subiendo {FILE_NAME} a GCS...")
storage_client = storage.Client(credentials=credentials, project=PROJECT_ID)
bucket = storage_client.bucket(GCS_BUCKET)
blob = bucket.blob(FILE_NAME)

blob.upload_from_filename(LOCAL_FILE_PATH)
print(f"✅ Archivo en la nube: {GCS_URI}")

# --- 4. PASO 2: VINCULAR BIGQUERY (EXTERNAL TABLE) ---
print(f"🔗 Vinculando BigQuery a GCS...")
bq_client = bigquery.Client(credentials=credentials, project=PROJECT_ID)

query_create_external = f"""
CREATE OR REPLACE EXTERNAL TABLE `{PROJECT_ID}.pienza_big.synthetic_manifold_v8_downscaled`
OPTIONS (
  format = 'PARQUET',
  uris = ['{GCS_URI}']
);
"""

try:
    query_job = bq_client.query(query_create_external)
    query_job.result()
    print("\n🏆 ¡MISIÓN CUMPLIDA!")
    print(f"Tabla externa lista en: {PROJECT_ID}.pienza_big.synthetic_manifold_v8_downscaled")
    print("Streamlit tiene luz verde para el despegue.")

except Exception as e:
    print("\n❌ ERROR CRÍTICO:")
    print(e)

📦 Subiendo 06XX_260503_synthetic_manifold_v8_downscaled.parquet a GCS...


✅ Archivo en la nube: gs://pienza-streamlit/06XX_260503_synthetic_manifold_v8_downscaled.parquet
🔗 Vinculando BigQuery a GCS...

🏆 ¡MISIÓN CUMPLIDA!
Tabla externa lista en: drivers-dilemma.pienza_big.synthetic_manifold_v8_downscaled
Streamlit tiene luz verde para el despegue.


In [15]:
# ==============================================================================
# CELL: THE FINAL SEMANTIC RE-FORGE (MICRO-RESOLUTION DICTIONARIES)
# Purpose: Generate the final dim parquets for Streamlit using Downscaled names.
# ==============================================================================
import pandas as pd
import os

print("🎨 Re-forging Dictionaries with Micro-Resolution Names...")
print("-" * 65)

# --- 1. CONFIGURACIÓN DE RUTAS (SOP 06XX) ---
DUMP_DIR = "/workspaces/pienza/data/dumped_files"
SOP_PREFIX = "06XX"
FECHA_HOY = "260503"  # 3 de Mayo, 2026

# --- 2. RE-FORGE PICKUP DICTIONARY (ORIGINS) ---
# Extraemos el mapeo único del ID de GAN al Nombre Downscaled
df_dim_pick_final = df_synthetic[['pickup_id_GAN', 'pickup_name_down']].drop_duplicates()
df_dim_pick_final.columns = ['zone_key', 'semantic_name']

# --- 3. RE-FORGE DROPOFF DICTIONARY (DESTINATIONS) ---
df_dim_drop_final = df_synthetic[['dropoff_id_GAN', 'dropoff_name_down']].drop_duplicates()
df_dim_drop_final.columns = ['zone_key', 'semantic_name']

# --- 4. CLEANING: VVS1 UI Experience ---
def polish_names(name):
    if pd.isna(name) or name == "unassigned_area": 
        return "Unassigned"
    # Reemplazamos guiones bajos por espacios y aplicamos Title Case
    return str(name).replace('_', ' ').title().strip()

df_dim_pick_final['semantic_name'] = df_dim_pick_final['semantic_name'].apply(polish_names)
df_dim_drop_final['semantic_name'] = df_dim_drop_final['semantic_name'].apply(polish_names)

# --- 5. PERSISTENCIA LOCAL ---
PICK_SAVE_PATH = f"{DUMP_DIR}/{SOP_PREFIX}_{FECHA_HOY}_dim_pickup_micro.parquet"
DROP_SAVE_PATH = f"{DUMP_DIR}/{SOP_PREFIX}_{FECHA_HOY}_dim_dropoff_micro.parquet"

os.makedirs(os.path.dirname(PICK_SAVE_PATH), exist_ok=True)

df_dim_pick_final.to_parquet(PICK_SAVE_PATH, index=False)
df_dim_drop_final.to_parquet(DROP_SAVE_PATH, index=False)

print(f"✅ DICTIONARIES RE-FORGED:")
print(f"   📍 Pickups:  {len(df_dim_pick_final)} Micro-Polygons -> {PICK_SAVE_PATH}")
print(f"   🏁 Dropoffs: {len(df_dim_drop_final)} Micro-Polygons -> {DROP_SAVE_PATH}")
print("-" * 65)
print("🚀 UI-Ready: Los nombres ahora son granulares y legibles.")

🎨 Re-forging Dictionaries with Micro-Resolution Names...
-----------------------------------------------------------------
✅ DICTIONARIES RE-FORGED:
   📍 Pickups:  98 Micro-Polygons -> /workspaces/pienza/data/dumped_files/06XX_260503_dim_pickup_micro.parquet
   🏁 Dropoffs: 119 Micro-Polygons -> /workspaces/pienza/data/dumped_files/06XX_260503_dim_dropoff_micro.parquet
-----------------------------------------------------------------
🚀 UI-Ready: Los nombres ahora son granulares y legibles.


In [16]:
import os
from google.cloud import storage
from google.oauth2 import service_account

print("☁️ Iniciando subida de Diccionarios Semánticos a GCS...")
print("-" * 65)

# --- 1. CONFIGURACIÓN ---
PROJECT_ID = "drivers-dilemma"
SERVICE_ACCOUNT_JSON = '/workspaces/pienza/secrets/service-account.json'
GCS_BUCKET = "pienza-streamlit"
DUMP_DIR = "/workspaces/pienza/data/dumped_files"
SOP_PREFIX = "06XX"
FECHA_HOY = "260503"

# Definimos los archivos locales que queremos subir
files_to_upload = [
    f"{SOP_PREFIX}_{FECHA_HOY}_dim_pickup_micro.parquet",
    f"{SOP_PREFIX}_{FECHA_HOY}_dim_dropoff_micro.parquet"
]

# --- 2. CREDENCIALES Y CLIENTE ---
credentials = service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_JSON)
storage_client = storage.Client(credentials=credentials, project=PROJECT_ID)
bucket = storage_client.bucket(GCS_BUCKET)

# --- 3. BATCH UPLOAD ---
for file_name in files_to_upload:
    local_path = os.path.join(DUMP_DIR, file_name)
    
    if os.path.exists(local_path):
        print(f"⏳ Subiendo: {file_name}...")
        blob = bucket.blob(file_name)
        blob.upload_from_filename(local_path)
        print(f"   ✅ Exitoso: gs://{GCS_BUCKET}/{file_name}")
    else:
        print(f"   ⚠️ Error: No se encontró el archivo local en {local_path}")

print("-" * 65)
print("🚀 SINCRONIZACIÓN COMPLETA. Los diccionarios están listos para la UI.")

☁️ Iniciando subida de Diccionarios Semánticos a GCS...
-----------------------------------------------------------------
⏳ Subiendo: 06XX_260503_dim_pickup_micro.parquet...
   ✅ Exitoso: gs://pienza-streamlit/06XX_260503_dim_pickup_micro.parquet
⏳ Subiendo: 06XX_260503_dim_dropoff_micro.parquet...
   ✅ Exitoso: gs://pienza-streamlit/06XX_260503_dim_dropoff_micro.parquet
-----------------------------------------------------------------
🚀 SINCRONIZACIÓN COMPLETA. Los diccionarios están listos para la UI.
